# Precision & Memory — see the formats yourself

A short companion to the lesson [*Precision & Memory*](https://lms-p-45c03.web.app/topics/math-infra/precision-and-memory/).
Cast a number into each float format and **see** — in charts — the **byte width**, the **range** (where
fp16 overflows but bf16 holds), the **precision** (bf16 is coarser), and **fp4**'s tiny world.

**Runs on CPU** — `numpy`, `ml_dtypes` (a JAX dependency Colab already has), and `matplotlib`. No calculus, no GPU.

## 1. Setup

In [ ]:
# Colab has these; if not:  !pip install -q ml_dtypes matplotlib
import numpy as np, ml_dtypes
import matplotlib.pyplot as plt
np.seterr(over="ignore", invalid="ignore")     # we trigger overflow on purpose

bf16 = ml_dtypes.bfloat16
fp4  = ml_dtypes.float4_e2m1fn                  # OCP "E2M1": 1 sign + 2 exponent + 1 mantissa bit
print("numpy", np.__version__, "· ml_dtypes", ml_dtypes.__version__)

## 2. Bytes are the lever
A number's byte width is what it costs to move on the memory bus. Halving it ~doubles **arithmetic
intensity** (FLOPs per byte) — which is *why* lower precision keeps the matrix unit fed.

In [ ]:
fmts = [("fp32", 4.0), ("fp16", 2.0), ("bf16", 2.0), ("fp4", 0.5)]
for n, b in fmts:
    print(f"{n}: {b} bytes / number")

plt.figure(figsize=(6, 3))
bars = plt.bar([n for n, _ in fmts], [b for _, b in fmts],
               color=["#94a3b8", "#dc2626", "#2563eb", "#7c3aed"])
plt.bar_label(bars, fmt="%.1f B"); plt.ylabel("bytes / number")
plt.title("Byte width — the throughput lever"); plt.tight_layout(); plt.show()

## 3. The overflow headline
fp16 keeps only a **5-bit exponent**, so it caps at ~65,504 and a big activation overflows to `inf`.
bf16 kept fp32's **8-bit exponent** (~3.4e38 range), so it holds. That range is why ML picked bf16.

In [ ]:
big = np.float32(1e5)                        # a big activation value
print("cast 1e5 ->   bf16:", float(bf16(big)), "  |  fp16:", float(np.float16(big)))

def maxfinite(dt):
    try:    return float(ml_dtypes.finfo(dt).max)      # bf16 / fp4
    except Exception: return float(np.finfo(dt).max)   # fp16 / fp32
print("max finite ->  fp16:", maxfinite(np.float16),
      " | bf16:", f"{maxfinite(bf16):.1e}", " | fp4:", maxfinite(fp4))

## 4. The whole trade in one chart
For every magnitude from 1e-8 to 1e8, cast it and plot the **rounding error** — the line only shows
where the format is usable (it stops where the value overflows, underflows, or rounds beyond ~50%).
You see **range** (how far the line reaches) and **precision** (how low the error sits) at once.

In [ ]:
xs = np.logspace(-8, 8, 400)
def errband(dt):
    out = []
    for x in xs:
        r = float(dt(np.float64(x)))
        if not np.isfinite(r) or r == 0: out.append(np.nan); continue
        e = abs(r - x) / x * 100
        out.append(e if e <= 50 else np.nan)            # >50% = effectively can't represent
    return np.array(out)

plt.figure(figsize=(8, 4.5))
for n, dt, c in [("fp32", np.float32, "#94a3b8"), ("bf16", bf16, "#2563eb"),
                 ("fp16", np.float16, "#dc2626"), ("fp4 (E2M1)", fp4, "#7c3aed")]:
    plt.plot(xs, errband(dt), label=n, lw=2.5, color=c)
plt.axvline(65504, ls="--", color="#dc2626", alpha=.5)
plt.text(65504, 0.0005, " fp16 cliff (~65k)", color="#dc2626", fontsize=8, va="bottom")
plt.xscale("log"); plt.yscale("log")
plt.xlabel("magnitude of the value"); plt.ylabel("rounding error  (%)")
plt.title("Range (how far the line reaches) vs precision (how low it sits)")
plt.legend(); plt.grid(alpha=.25, which="both"); plt.tight_layout(); plt.show()
print("bf16 spans the whole axis but sits high (coarse); fp16 sits low (fine) but stops at ~65k;")
print("fp4 exists only in a tiny window, very coarse — which is why fp4 needs value scaling.")

## 5. fp4 (E2M1) — the tiny world
Four bits can represent only a handful of magnitudes. Real values **snap** to the nearest dot — which
is why fp4 only works after values are **scaled into range** first.

In [ ]:
reps = sorted({float(fp4(v)) for v in np.arange(0, 6.01, 0.02)})
print("fp4 can ONLY represent:", reps)
for v in [0.1, 1.7, 5.0, 7.0]:
    print(f"  {v}  ->  {float(fp4(v))}")

plt.figure(figsize=(8, 1.8))
plt.scatter(reps, [0]*len(reps), s=90, color="#7c3aed", zorder=3)
for r in reps: plt.text(r, 0.12, str(r), ha="center", fontsize=8)
plt.scatter([0.1, 1.7, 5.0, 7.0], [-0.12]*4, marker="v", color="#dc2626", zorder=3)
for v in [0.1, 1.7, 5.0, 7.0]:
    plt.annotate("", xy=(float(fp4(v)), 0), xytext=(v, -0.12),
                 arrowprops=dict(arrowstyle="->", color="#dc2626", alpha=.6))
plt.ylim(-0.4, 0.4); plt.yticks([]); plt.xlim(-0.3, 7.3)
plt.title("fp4 (E2M1): every value snaps to one of 8 magnitudes"); plt.tight_layout(); plt.show()

## Put it together
1. **Bytes (§2):** fp32 → bf16 halves the bytes. What does that do to arithmetic intensity, and why does it keep the MXU busy?
2. **Range (§3–4):** in the chart, which line reaches furthest right, and which stops first — and which bits decide that?
3. **Precision (§4):** bf16's line sits *above* fp16's (more error). So why does training prefer bf16 anyway?
4. **fp4 (§5):** the red arrows show values snapping to dots. Why can't you cast raw model values to fp4 — what has to happen first?

Back to the lesson → [Precision & Memory](https://lms-p-45c03.web.app/topics/math-infra/precision-and-memory/)